# 3.1 幾個位置的資訊，怎樣按比例混在一起？


兩個位置的數字卡分別是 `[2,0]` 和 `[0,4]`。希望多用第一張時，可以取第一張九成、第二張一成：第一格得 `0.9×2+0.1×0=1.8`，第二格得 `0.9×0+0.1×4=0.4`。

這叫**加權平均**。每張卡乘自己的比例，再逐格相加；比例非負、總和為 1，結果便是各卡的混合。它與把卡片原封拼接不同：兩張兩格卡混完，只剩一張兩格卡 `[1.8,0.4]`。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

values = torch.tensor([[2.0, 0.0], [0.0, 4.0]])
weights = torch.tensor([0.9, 0.1])
output = weights @ values
print(output)
assert torch.allclose(output, torch.tensor([1.8, 0.4]))

`values` 每列是一個位置，`weights` 為每位置指定比例；`@` 把相同特徵欄加權相加。輸出與手算一致，斷言核對兩格。

**注意力**（attention）的核心讀取動作就是這種按比例混合。這裡還是由人指定 0.9、0.1；下一節才讓內容決定比例。注意力不一定只挑一個位置，本例兩張都有參與。

比例較大也不直接代表帶回數值較大。例如同一格中 `0.9×1=0.9`，而 `0.1×100=10`。需要一起看比例與內容；更不能只憑比例宣佈模型理解某個概念。本節的兩格沒有固定語意，只是手算材料。

練習交換比例成 `[0.1,0.9]`，預期輸出 `[0.2,3.6]`，將斷言同步修改再核對。你改的是讀法，兩張來源卡沒有改。

<details>
<summary>補充與重做</summary>

一般 Linear 權重可負、也不必加總為 1；這裡的讀取比例遵守機率式分配。矩陣與特徵軸見 [W.3](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)、[W.5](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.5)。`allclose` 只作數值核對，本例沒有可調參數或訓練。

</details>
